# 🧩 Integración: Gradio y MediaPipe

**Materiales desarrollados por Matías Barreto, 2026**  
**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**  
* **Nomenclatura Oficial:** Procesamiento Digital de Imágenes  
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital  

---

## Objetivo

En este laboratorio vamos a conectar dos herramientas que ya conocemos: **MediaPipe** para detectar puntos clave en imágenes y **Gradio** para construir interfaces web interactivas. El resultado va a ser una aplicación que cualquier persona puede usar desde el navegador, sin instalar nada.

## Al terminar este material vamos a poder:

1. Explicar qué es una **Skill** y por qué empaquetar conocimiento de forma estructurada tiene sentido.
2. Construir una interfaz web mínima con `gr.Interface` a partir de una función Python.
3. Crear layouts con control total usando `gr.Blocks`, botones y eventos.
4. Integrar un detector de MediaPipe dentro de una interfaz Gradio lista para compartir.

## Microglosario

Antes de escribir código, definimos los términos clave.

| Término | Definición | Analogía |
|---|---|---|
| **Skill** | Paquete de conocimiento estructurado y reutilizable que un agente puede cargar automáticamente | Como un libro de recetas profesional: no explicás cada vez cómo hacer una bechamel; la receta está ahí, lista para quien la necesite |
| **`gr.Interface`** | Forma rápida de envolver una función Python en una interfaz web con entradas y salidas definidas | Como enmarcar un cuadro: le das la obra (la función) y el marco se encarga de que se vea bien |
| **`gr.Blocks`** | Constructor de interfaces con control total sobre el layout, botones y eventos | Como armar un tablero de control a medida: vos elegís dónde va cada componente y qué hace |
| **Componente** | Elemento de interfaz (imagen, texto, slider) que conecta al usuario con la función Python | Como los enchufes de un equipo de audio: cada uno tiene un tipo de señal específico |
| **Evento** | Acción del usuario que dispara la ejecución de una función (click, cambio de valor, envío) | Como el timbre de la puerta: cuando suena, alguien va a atender |

## ✦ Concepto: ¿Qué es una Skill?

Antes de arrancar con el código, vamos a entender qué problema resuelve el concepto de **Skill** y por qué Hugging Face lo adoptó como estándar en su ecosistema de agentes.

### El problema: el agente no tiene contexto

Imaginemos que le pedimos a un agente de IA que publique un modelo en Hugging Face. Sin instrucciones específicas, el agente va a:

- No saber cómo autenticarse con nuestro token
- No conocer el formato de datos que usamos
- Publicar el modelo sin README, sin licencia, sin ejemplos de uso

Esos errores no son de inteligencia: son de **falta de contexto de dominio**.

### Primera respuesta: el prompt largo

Una reacción natural sería escribir un prompt detallado con todas las instrucciones:

```
Sos un experto en Hugging Face. Cuando publiques un modelo:
1. Autenticáte con HF_TOKEN
2. Validá que el dataset esté en formato CSV con columnas: text, label, split
3. Entrená con learning_rate=2e-5, batch_size=32, epochs=3
4. Creá un model card con descripción, métricas y licencia CC-BY-4.0
...
[continúa por cientos de líneas]
```

El problema es que ese prompt:

- ❌ No es reutilizable: hay que pegarlo en cada conversación nueva
- ❌ No es compartible: cada persona del equipo tiene su copia desactualizada
- ❌ No es mantenible: si cambia algo, hay que actualizar en decenas de lugares
- ❌ No tiene versiones: imposible saber qué cambió entre iteraciones

### La solución: la Skill

Una **Skill** empaqueta ese mismo conocimiento en un formato estructurado con un archivo `SKILL.md`:

```
mi-skill/
└── SKILL.md    ← metadatos + instrucciones
```

El archivo comienza con metadatos en YAML y luego contiene las instrucciones:

```yaml
---
name: "gradio-mediapipe"
description: "Construir interfaces web para modelos de visión con Gradio y MediaPipe."
---
# Instrucciones detalladas para el agente...
```

**Beneficios concretos:**

- ✓ Un solo lugar para mantener el conocimiento
- ✓ Versionable con git
- ✓ Compartible con todo el equipo
- ✓ Los agentes la cargan automáticamente cuando la necesitan

> El archivo `SKILL.md` que define el estilo de estos materiales *es en sí mismo una Skill*. Ya la estuvimos usando sin darnos cuenta.

*Fuente: [🤗 Context Course — Unit 1: What Are Skills?](https://huggingface.co/learn/context-course/unit1/what-are-skills)*

## Paso 1 — Instalación de herramientas

Vamos a instalar las tres bibliotecas que necesitamos para este notebook.

In [ ]:
# Instalamos las herramientas necesarias
# El símbolo ! indica que el comando se ejecuta en la terminal, no en Python
# --quiet reduce la cantidad de texto que muestra la instalación
!pip install gradio mediapipe opencv-python-headless --quiet

# Verificamos que todo se importa correctamente
import gradio as gr
import mediapipe as mp
import cv2
import numpy as np

version_gradio = gr.__version__
version_mediapipe = mp.__version__
version_opencv = cv2.__version__
version_numpy = np.__version__

print("✓ Entorno listo.")
print(f"  gradio     {version_gradio}")
print(f"  mediapipe  {version_mediapipe}")
print(f"  opencv     {version_opencv}")
print(f"  numpy      {version_numpy}")

## Paso 2 — Patrón `gr.Interface`: envolver una función en segundos

`gr.Interface` es el camino más corto de una función Python a una interfaz web. Solo necesita tres argumentos:

1. `fn` — la función que procesa la entrada
2. `inputs` — el tipo de componente de entrada (imagen, texto, número...)
3. `outputs` — el tipo de componente de salida

### ✦ Paso manual: el flujo a mano

Antes de ejecutar, pensemos cómo funciona el recorrido completo:

```
Usuario sube imagen desde el navegador
         ↓
  gr.Interface la convierte a un array NumPy (H × W × 3, RGB)
         ↓
  Llama a nuestra función con ese array
         ↓
  La función devuelve un nuevo array NumPy (imagen procesada)
         ↓
  gr.Interface muestra el resultado en el navegador
```

Vamos a probar con un filtro de escala de grises: la función recibe la imagen en color y devuelve la misma imagen en grises, convertida de vuelta a 3 canales para que Gradio la muestre correctamente.

In [ ]:
import gradio as gr
import numpy as np
import cv2

def aplicar_filtro_gris(imagen_entrada):
    """
    Recibe una imagen en formato NumPy (alto x ancho x 3, RGB).
    Devuelve la misma imagen convertida a escala de grises, en 3 canales.
    """
    # Convertimos la imagen a un solo canal de grises
    imagen_gris_un_canal = cv2.cvtColor(imagen_entrada, cv2.COLOR_RGB2GRAY)

    # Gradio espera imágenes con 3 canales (RGB)
    # Apilamos el canal gris tres veces para cumplir ese formato
    canal_rojo = imagen_gris_un_canal
    canal_verde = imagen_gris_un_canal
    canal_azul = imagen_gris_un_canal
    imagen_gris_tres_canales = np.stack([canal_rojo, canal_verde, canal_azul], axis=-1)

    return imagen_gris_tres_canales


# Construimos la interfaz: tres argumentos son suficientes
interfaz_filtro = gr.Interface(
    fn=aplicar_filtro_gris,
    inputs=gr.Image(label="Imagen original"),
    outputs=gr.Image(label="Imagen en grises"),
    title="Filtro de escala de grises",
    description="Subí una fotografía. La aplicación la va a convertir a escala de grises.",
    flagging_mode="never"
)

# Lanzamos la interfaz — se abre un link en la celda de salida
interfaz_filtro.launch()

## Paso 3 — Patrón `gr.Blocks`: control total del layout

`gr.Blocks` nos da control completo sobre el diseño. Podemos definir exactamente dónde aparece cada componente, conectar múltiples funciones a múltiples botones y crear interfaces con filas, columnas y pestañas.

| | `gr.Interface` | `gr.Blocks` |
|---|---|---|
| Velocidad de desarrollo | ✓ Muy rápido | Más verboso |
| Control del layout | Fijo | ✓ Total |
| Múltiples funciones | Una sola | ✓ Varias |
| Eventos personalizados | Limitados | ✓ Completos |

Vamos a construir una calculadora simple para ver cómo se conectan los componentes con los eventos.

In [ ]:
import gradio as gr

def calcular_resultado(numero_a, numero_b, operacion):
    """
    Recibe dos números y el nombre de la operación.
    Devuelve el resultado como texto.
    """
    if operacion == "Suma":
        resultado = numero_a + numero_b
    elif operacion == "Resta":
        resultado = numero_a - numero_b
    elif operacion == "Multiplicación":
        resultado = numero_a * numero_b
    elif operacion == "División":
        if numero_b == 0:
            return "Error: no se puede dividir por cero"
        resultado = numero_a / numero_b
    else:
        return "Operación no reconocida"

    texto_operacion = str(operacion)
    texto_resultado = f"{numero_a} — {texto_operacion} — {numero_b} = {resultado}"
    return texto_resultado


# Construimos la interfaz con gr.Blocks
# El bloque 'with' define el contenedor principal de la interfaz
with gr.Blocks(title="Calculadora") as interfaz_calculadora:

    gr.Markdown("## Calculadora interactiva")
    gr.Markdown("Ingresá dos números, elegí la operación y presioná Calcular.")

    # gr.Row agrupa componentes en una fila horizontal
    with gr.Row():
        entrada_numero_a = gr.Number(label="Número A", value=10)
        entrada_numero_b = gr.Number(label="Número B", value=5)

    # gr.Radio muestra opciones como botones de selección exclusiva
    selector_operacion = gr.Radio(
        choices=["Suma", "Resta", "Multiplicación", "División"],
        label="Operación",
        value="Suma"
    )

    boton_calcular = gr.Button("Calcular")

    salida_resultado = gr.Textbox(label="Resultado")

    # Conectamos el botón con la función
    # inputs: los componentes cuyos valores se pasan a la función
    # outputs: el componente donde se muestra lo que devuelve la función
    boton_calcular.click(
        fn=calcular_resultado,
        inputs=[entrada_numero_a, entrada_numero_b, selector_operacion],
        outputs=salida_resultado
    )

interfaz_calculadora.launch()

## Paso 4 — Integración: MediaPipe dentro de Gradio

Ahora combinamos lo que ya conocemos. La función que detecta landmarks con MediaPipe se convierte en la función central de la interfaz Gradio.

El flujo es el mismo que en el Paso 2, pero ahora la función hace algo más interesante:

```
Usuario sube fotografía
         ↓
  Gradio la convierte a NumPy RGB
         ↓
  Nuestra función la procesa con Face Mesh de MediaPipe
  (478 puntos clave faciales)
         ↓
  Devolvemos la imagen con los landmarks dibujados
         ↓
  Gradio muestra la imagen anotada en el navegador
```

Usamos el **Face Mesh** de MediaPipe — el mismo modelo que vimos en el notebook `01_Detección_Puntos_Clave_Faciales`. Aquí lo integramos en una interfaz lista para compartir.

In [ ]:
import gradio as gr
import mediapipe as mp
import numpy as np

# ── Configuración del detector ──────────────────────────────────────────

# Accedemos al módulo Face Mesh y a las utilidades de dibujo
modulo_face_mesh = mp.solutions.face_mesh
modulo_dibujo = mp.solutions.drawing_utils
estilos_dibujo = mp.solutions.drawing_styles

# Creamos el detector con los parámetros que queremos
# static_image_mode=True: procesamos imágenes sueltas (no video en tiempo real)
# max_num_faces=2: detectamos hasta dos rostros por imagen
# refine_landmarks=True: puntos más precisos en ojos y labios
# min_detection_confidence=0.5: umbral mínimo para aceptar una detección
detector_facial = modulo_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=2,
    refine_landmarks=True,
    min_detection_confidence=0.5
)

print("✓ Detector de Face Mesh inicializado.")


# ── Función principal ────────────────────────────────────────────────────

def detectar_landmarks_faciales(imagen_entrada):
    """
    Recibe una imagen RGB como array NumPy.
    Detecta los 478 puntos clave faciales con MediaPipe Face Mesh.
    Devuelve la imagen con los landmarks dibujados.
    """
    # Procesamos la imagen con el detector
    # MediaPipe espera la imagen en formato RGB, que es lo que Gradio envía
    resultado_deteccion = detector_facial.process(imagen_entrada)

    # Trabajamos sobre una copia para no modificar la imagen original
    imagen_anotada = imagen_entrada.copy()

    # Si no se detectó ningún rostro, devolvemos la imagen sin cambios
    if resultado_deteccion.multi_face_landmarks is None:
        print("No se detectó ningún rostro en la imagen.")
        return imagen_anotada

    # Contamos los rostros detectados para informar al usuario
    cantidad_rostros = len(resultado_deteccion.multi_face_landmarks)
    print(f"Rostros detectados: {cantidad_rostros}")

    # Dibujamos los puntos clave de cada rostro detectado
    for puntos_rostro in resultado_deteccion.multi_face_landmarks:
        modulo_dibujo.draw_landmarks(
            image=imagen_anotada,
            landmark_list=puntos_rostro,
            connections=modulo_face_mesh.FACEMESH_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=estilos_dibujo.get_default_face_mesh_tesselation_style()
        )

    return imagen_anotada


# ── Interfaz Gradio ──────────────────────────────────────────────────────

interfaz_landmarks = gr.Interface(
    fn=detectar_landmarks_faciales,
    inputs=gr.Image(
        label="Fotografía",
        type="numpy"
    ),
    outputs=gr.Image(
        label="Landmarks detectados"
    ),
    title="Detector de Landmarks Faciales — Face Mesh",
    description=(
        "Subí una fotografía con uno o dos rostros. "
        "La aplicación va a detectar los 478 puntos clave del Face Mesh de MediaPipe "
        "y los va a dibujar sobre la imagen."
    ),
    flagging_mode="never"
)

interfaz_landmarks.launch(share=False)

print()
print("✓ Interfaz activa.")
print("  Abrí el link de arriba en el navegador para usar la aplicación.")

## Para explorar

Estas son las fuentes que usamos como base para este notebook:

- [🤗 Context Course — Unit 1: What Are Skills?](https://huggingface.co/learn/context-course/unit1/what-are-skills) — El concepto de Skill explicado con ejemplos de agentes
- [🤗 huggingface-gradio Skill (GitHub)](https://github.com/huggingface/skills/tree/main/skills/huggingface-gradio) — El Skill oficial de Hugging Face para construir interfaces Gradio
- [Documentación oficial de Gradio](https://www.gradio.app/docs) — Referencia completa de componentes, layouts y eventos
- [MediaPipe Face Landmarker](https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker) — Documentación del modelo de landmarks faciales

---

### ✎ Para pensar

1. **Sobre `gr.Interface` vs `gr.Blocks`:** Revisá el detector de landmarks que construimos. ¿Qué tendríamos que cambiar para agregar un segundo botón que, en lugar de detectar landmarks, aplique el filtro de grises del Paso 2? ¿Alcanzaría con `gr.Interface` o necesitaríamos `gr.Blocks`? ¿Por qué?

2. **Sobre el concepto de Skill:** Revisá el archivo `SKILL.md` que guía el estilo de estos materiales. ¿Qué ventajas concretas tiene tener ese conocimiento en un archivo separado en lugar de escribirlo en el prompt de cada conversación?

3. **Sobre la integración:** La función `detectar_landmarks_faciales` recibe y devuelve arrays NumPy. ¿Qué habría que cambiar para que la entrada fuera un video en lugar de una imagen? ¿Qué componente de Gradio usarías? ¿La función de procesamiento cambiaría en algo?